# Trace Logs: Bronze → Silver Transformation

Extracts Microsoft Log Analytics App Insights **trace** records from the Bronze Lakehouse,
parses embedded key-value fields out of the semi-structured `Message` column,
and writes the enriched dataset to the Silver Lakehouse.

## Target Schema
```
id | event_time | <expanded Message fields…> | Message
```

## Design Principles
* **Idempotent** – Re-running with the same date range replaces previously written data.
* **Parameterised** – `start_date` / `end_date` control which `event_time` window is processed.
* **Configurable field list** – Edit the *Configuration* cell to add or remove fields that should
  be promoted out of `Message` into first-class columns.

## 1 · Parameters

> **Fabric / Notebook parameters cell** – values here can be overridden at pipeline runtime
> via the *Set parameters* activity.

In [ ]:
# ---------------------------------------------------------------------------
# PARAMETERS  (override these at pipeline / job level as needed)
# ---------------------------------------------------------------------------

# Inclusive start of the event_time window to process (ISO-8601 date string)
start_date: str = "2024-01-01"

# Exclusive end of the event_time window to process
end_date: str = "2024-12-31"

# ---------------------------------------------------------------------------
# Lakehouse names (adjust to match your Fabric workspace)
# ---------------------------------------------------------------------------
bronze_lakehouse: str = "Payit_Ppd_Log_Analytics_LH"
silver_lakehouse: str = "SilverLakehouse"  # Update this to match your Silver lakehouse

# Full table paths (Tables/<name>) inside each Lakehouse
bronze_table: str = "appinsights_traces"          # Log Analytics trace export table in Bronze
silver_table: str = "AppTraces_Enriched" # Destination table in Silver

## 2 · Configuration – Fields to Extract from `Message`

Edit the list below to add or remove the fields you want promoted from `Message`
into independent columns.  Any field that is absent from a row's message will
result in a `NULL` for that column.

In [ ]:
# ---------------------------------------------------------------------------
# CONFIGURATION – list of field names to extract from the Message column
# ---------------------------------------------------------------------------
# Each entry is the exact key name as it appears in the log message, e.g.:
#   "OperationId=abc123"  →  field name "OperationId"
#
# The parser supports the following formats within a single message:
#   key=value
#   key="value with spaces"
#   key: value
#   key: "value with spaces"
#
# Add or remove entries freely – the notebook is schema-on-write.

FIELDS_TO_EXTRACT: list = [
    "OperationId",
    "ParentId",
    "UserId",
    "SessionId",
    "CorrelationId",
    "Environment",
    "ServiceName",
    "Version",
    "Severity",
    "Category",
    "RequestId",
    "TraceId",
    "SpanId",
    "MachineName",
    "ApplicationName",
]

## 3 · Setup – Imports and Utilities

In [ ]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
import re

# ---------------------------------------------------------------------------
# Helper – build a Spark regex-extract expression for a single field name.
# Patterns tried in order (first match wins):
#   1.  key="value"   or   key: "value"   (quoted)
#   2.  key=value     or   key: value     (unquoted, up to next whitespace / comma / pipe)
# ---------------------------------------------------------------------------

def _make_extract_col(field: str) -> F.Column:
    """
    Return a Column expression that extracts *field* from the ``Message`` column.

    Parameters
    ----------
    field : str
        The key name to search for in the message text.

    Returns
    -------
    pyspark.sql.Column
        A StringType column containing the extracted value, or NULL when absent.
    """
    # Escape the field name so it is safe to embed in a regex
    escaped = re.escape(field)

    # Pattern explanation:
    #   (?i)          – case-insensitive match
    #   {escaped}     – literal field name
    #   \s*[=:]\s*   – separator: '=' or ':' with optional surrounding whitespace
    #   (?:           – non-capturing group for two alternatives:
    #     "([^"]*)"  –   (1) double-quoted value
    #   |             –   OR
    #     ([^\s,|]*)  –   (2) unquoted value up to whitespace, comma, or pipe
    #   )
    pattern = rf'(?i){escaped}\s*[=:]\s*(?:"([^"]*)"|(\S+))'

    # regexp_extract(str, pattern, idx):
    #   group 1 → quoted value, group 2 → unquoted value
    # Combine: use quoted when non-empty, else fall back to unquoted.
    quoted   = F.regexp_extract(F.col("Message"), pattern, 1)
    unquoted = F.regexp_extract(F.col("Message"), pattern, 2)

    extracted = F.when(quoted != "", quoted).otherwise(
        F.when(unquoted != "", unquoted).otherwise(F.lit(None).cast(StringType()))
    )
    return extracted.alias(field)


def extract_message_fields(df: DataFrame, fields: list) -> DataFrame:
    """
    Expand the ``Message`` column of *df* by extracting *fields* as new columns.

    The new columns are inserted between ``event_time`` and ``Message`` so that
    the resulting schema follows the target layout::

        id | event_time | <field1> | <field2> | … | Message

    Parameters
    ----------
    df : DataFrame
        Source DataFrame containing at least ``id``, ``event_time``, and
        ``Message`` columns.
    fields : list[str]
        Names of keys to extract from ``Message``.

    Returns
    -------
    DataFrame
        New DataFrame with extracted columns appended after ``event_time``.
    """
    # Columns that must stay in their original positions
    keep_cols = [c for c in df.columns if c not in fields]

    # Build extraction expressions for every requested field
    extract_exprs = [_make_extract_col(f) for f in fields]

    # Append extracted columns to the dataframe
    df_with_fields = df.select(
        *[F.col(c) for c in keep_cols],
        *extract_exprs,
    )

    # Re-order so that expanded fields sit between event_time and Message
    base_cols = [c for c in keep_cols if c != "Message"]
    ordered   = base_cols + fields + ["Message"]
    # Only include columns that actually exist in the dataframe
    ordered   = [c for c in ordered if c in df_with_fields.columns]

    return df_with_fields.select(ordered)


print("✔ Utility functions loaded.")

## 4 · Read from Bronze Lakehouse

Reads only the rows in `[start_date, end_date)` from `event_time` to limit
the amount of data processed per run.

In [ ]:
# Microsoft Fabric Table Reference - Multiple Methods for Reliability
print(f"Reading Bronze table: {bronze_lakehouse}.{bronze_table}")
print(f"  event_time filter: [{start_date}, {end_date})")

# Try Method 1: Direct table reference (if lakehouse is attached)
try:
    print("Attempting Method 1: Direct table reference...")
    df_bronze = (
        spark.read.table(bronze_table)
        .filter(
            (F.col("event_time") >= F.lit(start_date).cast("timestamp"))
            & (F.col("event_time") < F.lit(end_date).cast("timestamp"))
        )
    )
    print("✅ Method 1 successful: Direct table reference")
    
except Exception as e1:
    print(f"❌ Method 1 failed: {e1}")
    
    # Try Method 2: SQL-based approach
    try:
        print("Attempting Method 2: SQL-based approach...")
        query = f"""
        SELECT * FROM {bronze_lakehouse}.{bronze_table}
        WHERE event_time >= '{start_date}' 
          AND event_time < '{end_date}'
        """
        df_bronze = spark.sql(query)
        print("✅ Method 2 successful: SQL-based approach")
        
    except Exception as e2:
        print(f"❌ Method 2 failed: {e2}")
        
        # Try Method 3: Check if lakehouse is in catalog and use three-part naming
        try:
            print("Attempting Method 3: Catalog-based lookup...")
            # List available databases to find the correct reference
            databases = [db.name for db in spark.catalog.listDatabases()]
            print(f"Available databases: {databases}")
            
            if bronze_lakehouse in databases:
                bronze_full_table = f"{bronze_lakehouse}.{bronze_table}"
                df_bronze = (
                    spark.read.table(bronze_full_table)
                    .filter(
                        (F.col("event_time") >= F.lit(start_date).cast("timestamp"))
                        & (F.col("event_time") < F.lit(end_date).cast("timestamp"))
                    )
                )
                print("✅ Method 3 successful: Catalog-based lookup")
            else:
                raise Exception(f"Lakehouse '{bronze_lakehouse}' not found in available databases: {databases}")
                
        except Exception as e3:
            print(f"❌ Method 3 failed: {e3}")
            raise Exception(
                f"All table access methods failed. Ensure:\n"
                f"1. Lakehouse '{bronze_lakehouse}' is attached to this notebook\n"
                f"2. Table '{bronze_table}' exists in the lakehouse\n"
                f"3. You have read permissions on the lakehouse\n"
                f"Errors: Method1={e1}, Method2={e2}, Method3={e3}"
            )

bronze_count = df_bronze.count()
print(f"  Rows read from Bronze: {bronze_count:,}")

# Quick schema check – warn if expected columns are missing
required_columns = {"id", "event_time", "Message"}
missing = required_columns - set(df_bronze.columns)
if missing:
    raise ValueError(
        f"Bronze table is missing expected column(s): {missing}. "
        "Verify the table name and schema."
    )

print(f"  Bronze schema: {df_bronze.dtypes}")

## 5 · Transform – Extract Fields from `Message`

In [ ]:
print(f"Extracting {len(FIELDS_TO_EXTRACT)} field(s) from Message: {FIELDS_TO_EXTRACT}")

df_silver = extract_message_fields(df_bronze, FIELDS_TO_EXTRACT)

print("\nResulting Silver schema:")
df_silver.printSchema()

## 6 · Write to Silver Lakehouse (Idempotent)

Ensures the Silver schema exists, then writes the enriched data partitioned
by **`event_date`** (YYYY-MM-DD). Dynamic partition overwrite mode ensures
only the partitions present in the current batch are replaced, making the
operation idempotent: re-running with the same date range produces the same
result without duplicating data.

In [ ]:
# ------------------------------------------------------------------
# SOLUTION: Explicit lakehouse context switching for multi-lakehouse setup
# ------------------------------------------------------------------

print(f"Writing to Silver lakehouse: {silver_lakehouse}")
print(f"Target table: {silver_table}")

# Add a date partition column derived from event_time
df_to_write = df_silver.withColumn(
    "event_date", F.to_date(F.col("event_time"))
)

# CRITICAL: Set lakehouse context before creating table
print(f"✔ Setting lakehouse context to: {silver_lakehouse}")
spark.sql(f"USE {silver_lakehouse}")

# Verify we're in the correct context
current_db = spark.sql("SELECT current_database()").collect()[0][0]
print(f"✔ Current database context: {current_db}")

# Check if table exists in the current (Silver) lakehouse
table_exists = False
try:
    spark.sql(f"DESCRIBE TABLE {silver_table}")
    table_exists = True
    print(f"✔ Table {silver_table} exists in {current_db}")
except:
    print(f"✔ Table {silver_table} does not exist - will create new")

# Configure write strategy based on table existence
if table_exists:
    # For existing table: Use dynamic partition overwrite (idempotent)
    print("Using dynamic partition overwrite for existing table")
    spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
    (
        df_to_write.write
        .format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")  # Allow schema evolution
        .partitionBy("event_date")
        .saveAsTable(silver_table)  # Just table name, context already set
    )
    print(f"✅ Table {silver_table} updated with dynamic partitioning")
else:
    # For new table: Reset partition mode and create with schema overwrite
    print("Creating new table with schema overwrite")
    # CRITICAL: Reset dynamic partition overwrite mode for new table creation
    spark.conf.set("spark.sql.sources.partitionOverwriteMode", "static")
    (
        df_to_write.write
        .format("delta")
        .mode("overwrite")
        .partitionBy("event_date")
        .saveAsTable(silver_table)  # Just table name, context already set
    )
    print(f"✅ Table {silver_table} created in {current_db}")

print(f"  Write complete. Rows written: {bronze_count:,}")

# Verify the table location
tables_in_silver = [row.tableName for row in spark.sql("SHOW TABLES").collect()]
if silver_table in tables_in_silver:
    print(f"✅ VERIFICATION: Table {silver_table} confirmed in {current_db}")
else:
    print(f"⚠️  WARNING: Table not found. Available tables: {tables_in_silver}")

## 7 · Validation & Summary

In [ ]:
df_validate = (
    spark.read.table(silver_full_table)
    .filter(
        (F.col("event_time") >= F.lit(start_date).cast("timestamp"))
        & (F.col("event_time") < F.lit(end_date).cast("timestamp"))
    )
)

silver_count = df_validate.count()

print("=" * 60)
print("RUN SUMMARY")
print("=" * 60)
print(f"  Date range processed : {start_date}  →  {end_date}")
print(f"  Bronze rows read     : {bronze_count:,}")
print(f"  Silver rows confirmed: {silver_count:,}")
print(f"  Fields extracted     : {FIELDS_TO_EXTRACT}")
print("=" * 60)

if bronze_count != silver_count:
    raise AssertionError(
        f"Row count mismatch: Bronze={bronze_count:,}, Silver={silver_count:,}. "
        "Investigate pipeline logs."
    )

print("✔ Validation passed.")

# Show a sample of the enriched output
display(df_validate.limit(10))